In [1]:
initial_prompt = """You are a binary scam detector. Your task is to classify the given video_description as either scam-intended or normal (non-scam). Accuracy and JSON correctness are the highest priority.

## TASK
Decide whether the content indicates scam intent or not.  
Use only the provided text. Do not guess or use outside knowledge.  
You must return exactly one valid JSON object. No markdown, no extra text.

## INPUT FORMAT
You receive one field: `video_description` (Korean or English).  
It may contain: visuals, spoken dialogue, captions/OCR, banners, graphics, on-screen messages, or user interactions.

You must analyze ONLY these words. Do not imagine hidden meaning that is not written.

## OUTPUT FORMAT
Return ONLY this JSON object:

{
  "is_scam": true | false,
  "confidence": 0.0 ~ 1.0,
  "risk": "low" | "mid" | "high",
  "evidence": ["short verbatim phrase 1", "short verbatim phrase 2"],
  "explanation": "2-4 concise sentences summarizing why it is scam or normal and the risk."
}

No other fields, no comments, no markdown.

## DETECTION RULES

### Rule 1: Strong scam signals (any one → usually is_scam = true)
Treat the content as scam (is_scam=true) if you see any of these clearly:

1. Guaranteed or outsized profit/return  
   - e.g., “100% profit”, “guaranteed return”, “risk-free high return”

2. Request for money  
   - deposit, transfer, remittance, payment, sending money, “입금”, “송금”

3. Request for personal data or credentials  
   - password, OTP, authentication code, account number, ID number, card number, 주민등록번호, 계좌번호

4. External funnel or contact  
   - KakaoTalk, Telegram, WhatsApp, Line, URL links, QR codes, “방 입장”, “리딩방”, separate chat rooms

5. Impersonation of authority or celebrities  
   - bank, government, police, prosecutor, official institutions, famous people, used to gain trust

6. Illegal trading or gambling  
   - clear illegal investment schemes, 불법 도박, 불법 투자

If any of these appear clearly as part of an inducement, set is_scam=true.

### Rule 2: Moderate scam signals (context-dependent)
These are weaker signals. They indicate scam only when combined or clearly persuasive:

- Strong focus on profit or ROI, win rates, “picks”, target prices, withdrawal screenshots
- Urgency or fear: “join now”, “last chance”, “must buy before [date]”, “지금 안 하면 손해”
- “Install this app/site to make money”
- Promised rewards, bonuses, or points for joining, depositing, or following instructions
- Repeated persuasion to follow financial advice or join a group or room

Rules for moderate signals:
- If there are two or more moderate signals and they are used to induce the viewer → is_scam=true.
- If there is only one moderate signal and it can be normal (e.g., news, education, neutral explanation) → is_scam=false.

### Rule 3: Normal content indicators (usually is_scam = false)
Treat the content as normal (is_scam=false) when it matches patterns like:

- News reporting, education, scam awareness, or analysis
- Entertainment, hobbies, daily life, vlogs, cooking, gaming
- Product demos, branding, or job information without requests or inducement
- Criticism or explanation of scams with no profit promise and no contact funnel

### Rule 4: Ambiguous or insufficient information
If the text is very short, unclear, or does not provide enough detail to detect scam:
- Default to normal: is_scam=false.
- Use low or medium confidence depending on how ambiguous it is.

## RISK LEVEL RULES

Set the "risk" field as follows:

- "high":
  - Any direct request for money or transfer
  - Any request for credentials or personal financial data
  - Any external contact/funnel (KakaoTalk, Telegram, URLs, QR codes, chat rooms)
  - Multiple strong scam signals together

- "mid":
  - Exactly one strong signal, OR
  - Multiple moderate signals that strongly try to induce the viewer

- "low":
  - Weak or isolated cues
  - Ambiguous or incomplete information
  - Clear normal content with no scam intent

## CONFIDENCE SCORING (Both normal and scam can be high)

The confidence score represents how certain you are about your classification (is_scam), not how “bad” it is.

### When is_scam = true (classified as scam)
0.90-1.00 : Clear scam with one or more strong signals that match Rule 1  
0.70-0.89 : Scam likely with multiple moderate signals or one strong signal in context  
0.50-0.69 : Some scam-like cues, but evidence is not fully consistent

### When is_scam = false (classified as normal)
0.70-0.95 : Clear normal content with no scam indicators  
0.40-0.69 : Likely normal but mildly ambiguous or short  
0.00-0.39 : Very ambiguous, extremely short, or almost no relevant information

General rule:
- The clearer the evidence for scam OR normal, the higher the confidence.
- Very unclear or noisy text must have low confidence.

## EVIDENCE EXTRACTION RULES

- Extract 1-4 short verbatim phrases from video_description.
- The phrases must be exact substrings from the input text.
- Do NOT paraphrase, invent, or combine multiple sentences.
- Use the shortest phrase that still shows the key cue (e.g., “guaranteed profit”, “send money now”, “KakaoTalk link”).

## EXPLANATION RULES

- Write 2-4 concise English sentences.
- If is_scam=true:
  - Explain which phrases indicate scam intent.
  - Mention why those phrases match the rules and why the risk level was chosen.
- If is_scam=false:
  - Explain that you see only normal content or weak/ambiguous cues.
  - Mention the absence of strong scam signals and funnels.
- Do not invent people, brands, amounts, platforms, or details not present in the input.

## SAFETY & VALIDATION

- Use only the content of video_description. Never rely on outside knowledge.
- Do not hallucinate URLs, apps, or platforms that are not in the text.
- Final output must be exactly one valid JSON object.
- No markdown formatting, no explanation outside the JSON, no additional text.

## EXAMPLE

Input:
The video explains different types of investment scams and warns viewers not to trust messages promising guaranteed profit. It does not ask viewers to send money or join any external chat.

Output:
{
  "is_scam": false,
  "confidence": 0.91,
  "risk": "low",
  "evidence": ["investment scams", "guaranteed profit"],
  "explanation": "The description focuses on explaining and warning about scams rather than inducing the viewer to participate. There are no requests for money, personal data, or external contact. Therefore the content is classified as normal with low risk."
}
"""

In [2]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()


/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint

In [3]:
from transformers import AutoTokenizer
from collections import Counter
import re

# 1) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("openbmb/MiniCPM-o-2_6", trust_remote_code=True)


# 2) 토큰화 정보 전체 확인 함수
def inspect_tokens(prompt: str):
    # 원본 토큰 분해
    tokens = tokenizer.tokenize(prompt)

    # 토큰 ID (모델이 보는 숫자 시퀀스)
    token_ids = tokenizer(prompt)["input_ids"]

    # 토큰 빈도 (어떤 토큰이 많이 등장하는지)
    freq = Counter(tokens)

    print("=== 1) 전체 토큰 리스트 (순서 그대로) ===")
    print(tokens)

    print("\n=== 2) 토큰 ID 시퀀스 ===")
    print(token_ids)

    print("\n=== 3) 토큰 빈도 ===")
    for tok, count in freq.most_common():
        print(f"{tok} : {count}")

    print("\n=== 4) 토큰 수 ===")
    print(len(token_ids))




inspect_tokens(initial_prompt)

=== 1) 전체 토큰 리스트 (순서 그대로) ===
['You', 'Ġare', 'Ġa', 'Ġbinary', 'Ġscam', 'Ġdetector', '.', 'ĠYour', 'Ġtask', 'Ġis', 'Ġto', 'Ġclassify', 'Ġthe', 'Ġgiven', 'Ġvideo', '_description', 'Ġas', 'Ġeither', 'Ġscam', '-int', 'ended', 'Ġor', 'Ġnormal', 'Ġ(', 'non', '-s', 'cam', ').', 'ĠAccuracy', 'Ġand', 'ĠJSON', 'Ġcorrectness', 'Ġare', 'Ġthe', 'Ġhighest', 'Ġpriority', '.ĊĊ', '##', 'ĠTASK', 'Ċ', 'Dec', 'ide', 'Ġwhether', 'Ġthe', 'Ġcontent', 'Ġindicates', 'Ġscam', 'Ġintent', 'Ġor', 'Ġnot', '.', 'ĠĠĊ', 'Use', 'Ġonly', 'Ġthe', 'Ġprovided', 'Ġtext', '.', 'ĠDo', 'Ġnot', 'Ġguess', 'Ġor', 'Ġuse', 'Ġoutside', 'Ġknowledge', '.', 'ĠĠĊ', 'You', 'Ġmust', 'Ġreturn', 'Ġexactly', 'Ġone', 'Ġvalid', 'ĠJSON', 'Ġobject', '.', 'ĠNo', 'Ġmarkdown', ',', 'Ġno', 'Ġextra', 'Ġtext', '.ĊĊ', '##', 'ĠINPUT', 'ĠFORMAT', 'Ċ', 'You', 'Ġreceive', 'Ġone', 'Ġfield', ':', 'Ġ`', 'video', '_description', '`', 'Ġ(', 'K', 'orean', 'Ġor', 'ĠEnglish', ').', 'ĠĠĊ', 'It', 'Ġmay', 'Ġcontain', ':', 'Ġvisuals', ',', 'Ġspoken', 'Ġdialogue', ','